In [ ]:
!pip install opencv-python-headless dlib face-alignment scikit-image scipy -q

import os
if not os.path.exists('shape_predictor_68_face_landmarks.dat'):
    !wget -q http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
    !bunzip2 shape_predictor_68_face_landmarks.dat.bz2
print('✅ 설치 완료')

In [ ]:
import numpy as np
import cv2
from scipy.interpolate import RBFInterpolator

class TPSWarper:
    def __init__(self, src_lmk, dst_lmk, image_shape):
        self.src = src_lmk.astype(np.float32)
        self.dst = dst_lmk.astype(np.float32)
        self.H, self.W = image_shape[:2]
        self._build_map()

    def _build_map(self):
        interp_inv_x = RBFInterpolator(self.dst, self.src[:, 0], kernel='thin_plate_spline')
        interp_inv_y = RBFInterpolator(self.dst, self.src[:, 1], kernel='thin_plate_spline')

        yy, xx = np.mgrid[0:self.H, 0:self.W]
        grid = np.column_stack([xx.ravel(), yy.ravel()]).astype(np.float32)

        self.map_x = interp_inv_x(grid).reshape(self.H, self.W).astype(np.float32)
        self.map_y = interp_inv_y(grid).reshape(self.H, self.W).astype(np.float32)

    def warp(self, image):
        return cv2.remap(image, self.map_x, self.map_y,
                         cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)

print('✅ TPSWarper 정의 완료')

In [ ]:
import face_alignment

fa = face_alignment.FaceAlignment(
    face_alignment.LandmarksType.TWO_D, device='cuda'
)

def detect_landmarks_server(image_path):
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    preds = fa.get_landmarks(img_rgb)
    if preds is None:
        raise ValueError('얼굴 미검출')
    return preds[0].astype(np.float32), img

def detect_kps_from_landmarks(lmk):
    return np.array([
        lmk[36:42].mean(axis=0),
        lmk[42:48].mean(axis=0),
        lmk[30],
        lmk[48],
        lmk[54],
    ], dtype=np.float32)

def transfer_expression(src_lmk, tgt_lmk, exp_weight=1.0):
    src_center = src_lmk.mean(axis=0)
    tgt_center = tgt_lmk.mean(axis=0)
    src_scale = np.sqrt(((src_lmk - src_center)**2).mean())
    tgt_scale = np.sqrt(((tgt_lmk - tgt_center)**2).mean())
    scale_ratio = src_scale / (tgt_scale + 1e-8)
    aligned = (tgt_lmk - tgt_center) * scale_ratio + src_center
    return src_lmk * (1 - exp_weight) + aligned * exp_weight

print('✅ 함수 정의 완료')

In [ ]:
from google.colab import files
import matplotlib.pyplot as plt

print('소스 이미지 업로드')
uploaded = files.upload()
SOURCE_IMAGE_PATH = list(uploaded.keys())[0]

src_lmk, src_img = detect_landmarks_server(SOURCE_IMAGE_PATH)
src_kps = detect_kps_from_landmarks(src_lmk)
print(f'✅ 랜드마크 검출: {src_lmk.shape}, KPS: {src_kps.shape}')

In [ ]:
import base64, json

def encode_landmarks(lmk_68, kps_5, w, h, frame_id=0):
    return {
        'landmark': base64.b64encode(lmk_68.astype(np.float32).tobytes()).decode(),
        'kps': base64.b64encode(kps_5.astype(np.float32).tobytes()).decode(),
        'image_width': w, 'image_height': h, 'frame_id': frame_id
    }

def decode_landmarks(payload, src_img_shape):
    lmk = np.frombuffer(base64.b64decode(payload['landmark']), np.float32).reshape(68, 2)
    kps = np.frombuffer(base64.b64decode(payload['kps']), np.float32).reshape(5, 2)
    H, W = src_img_shape[:2]
    mw, mh = payload['image_width'], payload['image_height']
    lmk = lmk / [mw, mh] * [W, H]
    kps = kps / [mw, mh] * [W, H]
    return lmk.astype(np.float32), kps.astype(np.float32)

# 테스트용 표정 시뮬레이션 (실제 사용 시 모바일 payload로 교체)
tgt_lmk_sim = src_lmk.copy()
tgt_lmk_sim[48][1] -= 10   # 입꼬리 올리기 (웃음)
tgt_lmk_sim[54][1] -= 10
tgt_lmk_sim[57][1] += 8
tgt_kps_sim = detect_kps_from_landmarks(tgt_lmk_sim)

src_h, src_w = src_img.shape[:2]
payload = encode_landmarks(tgt_lmk_sim, tgt_kps_sim, src_w, src_h)
received_lmk, received_kps = decode_landmarks(payload, src_img.shape)

# 표정 합성 & TPS warping
merged_lmk = transfer_expression(src_lmk, received_lmk, exp_weight=1.0)
warper = TPSWarper(src_lmk, merged_lmk, src_img.shape)
warped = warper.warp(src_img)

# 결과 시각화
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(cv2.cvtColor(src_img, cv2.COLOR_BGR2RGB))
axes[0].set_title('소스 원본')
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(warped, cv2.COLOR_BGR2RGB))
axes[1].set_title('TPS 표정 전이 결과')
axes[1].axis('off')
plt.show()
print('✅ 완료')